# Notebook 07 — Deployment End-to-End Validation

Kiểm thử thật chuỗi **RAW INPUT → FEATURE ENGINEERING → MODEL_FEATURES → MODEL → PREDICTION**, đồng thời kiểm tra cluster/recommend endpoints và bốn tab Streamlit.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for candidate in Path.cwd().resolve().parents:
        if (candidate / "src").exists() and (candidate / "5.DATA").exists():
            ROOT = candidate
            break
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

import importlib.util
from fastapi.testclient import TestClient
from streamlit.testing.v1 import AppTest

from src.features import CLUSTER_FEATURES, RAW_INPUT_FEATURES, get_model_features
from src.prediction_policy import (
    FINAL_HOLDOUT_MAX_YEAR, OBSERVED_DATA_MAX_YEAR,
    PRODUCT_SUPPORT_END_YEAR, prediction_support_status,
)

MODEL_DIR = ROOT / "4.MODELS" / "hitradar_popularity"
SECONDARY_DIR = ROOT / "4.MODELS" / "hitradar_secondary"
DATA_PATH = ROOT / "5.DATA" / "processed" / "ml_ready_dataset.parquet"
metrics = json.loads((MODEL_DIR / "final_test_metrics.json").read_text(encoding="utf-8"))
pipeline = joblib.load(MODEL_DIR / "popularity_pipeline.joblib")
expected_features = get_model_features(include_engineered=metrics["include_engineered"], include_time=metrics["include_time"])
assert metrics["model_features"] == expected_features
estimator_name = pipeline.named_steps["model"].__class__.__name__
expected_estimator = {"Linear Regression":"LinearRegression", "Random Forest":"RandomForestRegressor", "XGBoost":"XGBRegressor"}[metrics["selection_winner_model"]]
assert estimator_name == expected_estimator
assert pipeline.named_steps["features"].fit_row_count_ == metrics["final_refit_rows"]
print("Locked final configuration:", metrics["selection_winner_experiment"], "/", metrics["selection_winner_model"])
print("Model feature count:", len(expected_features))

Project root: D:\Hitradar\hitradar-main


Locked final configuration: Engineered With-Time / XGBoost
Model feature count: 32


In [2]:
raw_data = pd.read_parquet(DATA_PATH)
raw_example = raw_data.loc[[raw_data.index[-1]], RAW_INPUT_FEATURES]
engineered = pipeline.named_steps["features"].transform(raw_example)
model_features_present = all(f in engineered.columns for f in expected_features)
prediction_raw = float(pipeline.predict(raw_example)[0])
prediction = float(np.clip(prediction_raw, 0, 100))
e2e = {"raw_columns_ok":raw_example.columns.tolist()==RAW_INPUT_FEATURES,
       "feature_engineering_ok":model_features_present,
       "model_features":len(expected_features), "prediction_raw":prediction_raw,
       "prediction_clipped":prediction, "status":"PASS"}
assert np.isfinite(prediction)
display(pd.DataFrame([e2e]))

,raw_columns_ok,feature_engineering_ok,model_features,prediction_raw,prediction_clipped,status
0,True,True,32,38.561779,38.561779,PASS


## FastAPI integration: prediction, cluster, recommendation

In [3]:
api_path = ROOT / "5.UNG_DUNG" / "5.1.backend_api" / "api.py"
spec = importlib.util.spec_from_file_location("hitradar_round4_api", api_path)
api_module = importlib.util.module_from_spec(spec); spec.loader.exec_module(api_module)
client = TestClient(api_module.app)
payload = raw_example.iloc[0].to_dict()
payload["explicit"] = bool(payload["explicit"])
for key in ("release_year","key","mode","time_signature","release_month"):
    payload[key] = int(payload[key])

health = client.get("/health")
pred_response = client.post("/predict", json=payload)
payload_2020 = dict(payload, release_year=2020)
payload_2026 = dict(payload, release_year=2026)
pred_2020 = client.post("/predict", json=payload_2020)
pred_2026 = client.post("/predict", json=payload_2026)
cluster_payload = {feature: payload[feature] for feature in CLUSTER_FEATURES}
cluster_response = client.post("/cluster", json=cluster_payload)
query_id = str(raw_data.iloc[0]["track_id"])
recommend_response = client.get(f"/recommend/{query_id}?n=5")
assert health.status_code == pred_response.status_code == pred_2020.status_code == pred_2026.status_code == cluster_response.status_code == recommend_response.status_code == 200
assert health.json()["model_ready"] and health.json()["cluster_ready"] and health.json()["recommender_ready"]
assert abs(pred_response.json()["predicted_popularity"] - prediction) < 0.001
assert pred_2020.json()["temporal_extrapolation"] is False
assert pred_2020.json()["prediction_support_status"] == "within_product_support"
assert pred_2026.json()["temporal_extrapolation"] is True
assert pred_2026.json()["support_note"]
assert pred_2026.json()["product_support_end_year"] == PRODUCT_SUPPORT_END_YEAR
assert pred_2026.json()["observed_data_max_year"] == OBSERVED_DATA_MAX_YEAR
assert pred_2026.json()["final_holdout_max_year"] == FINAL_HOLDOUT_MAX_YEAR
direct_2026 = float(np.clip(pipeline.predict(pd.DataFrame([payload_2026])[RAW_INPUT_FEATURES])[0], 0, 100))
assert abs(pred_2026.json()["predicted_popularity"] - direct_2026) < 0.001
assert query_id not in {r["track_id"] for r in recommend_response.json()["recommendations"]}
display(pd.DataFrame([health.json()]))
display(pd.DataFrame([pred_response.json()]))
display(pd.DataFrame([pred_2020.json(), pred_2026.json()]))
display(pd.DataFrame([cluster_response.json()]))
display(pd.DataFrame(recommend_response.json()["recommendations"]))

,status,model_ready,cluster_ready,recommender_ready,model_file_exists,cluster_file_exists,recommender_file_exists,model_name,selection_winner_experiment,raw_input_count,model_feature_count,load_errors
0,ready,True,True,True,True,True,True,XGBRegressor,Engineered With-Time,17,32,{}


,predicted_popularity,popularity_tier,model_name,engineered_feature_count,feature_count,prediction_support_status,temporal_extrapolation,support_note,train_end_year,product_support_end_year,observed_data_max_year,final_holdout_max_year
0,38.5618,emerging,XGBoost,14,32,within_product_support,False,The release year is within HitRadar's document...,2018,2020,2021,2021


,predicted_popularity,popularity_tier,model_name,engineered_feature_count,feature_count,prediction_support_status,temporal_extrapolation,support_note,train_end_year,product_support_end_year,observed_data_max_year,final_holdout_max_year
0,25.9825,low,XGBoost,14,32,within_product_support,False,The release year is within HitRadar's document...,2018,2020,2021,2021
1,25.9825,low,XGBoost,14,32,temporal_extrapolation,True,The model was trained through 2018. The produc...,2018,2020,2021,2021


,cluster,chosen_k,feature_count
0,0,3,10


,track_id,cosine_similarity
0,2YSuy9tg3xtb2sKhvMRT3b,0.950803
1,5Nn2Dj7OQsGL6pgQ9iIzPp,0.947264
2,06DXs2hBRdjNs1qE1iYCQQ,0.946462
3,7vGxMdSqL9dSZsoSnOnLL6,0.940326
4,6BWRvw630R8z2vNMok6quI,0.934900


## Streamlit integration: bốn tab

In [4]:
streamlit_path = ROOT / "5.UNG_DUNG" / "5.2.frontend" / "streamlit_app.py"
app_test = AppTest.from_file(str(streamlit_path)).run(timeout=40)
tab_labels = [tab.label for tab in app_test.tabs]
release_year_input = next(item for item in app_test.number_input if item.label == "Release year")
warnings_2020 = [warning.value for warning in app_test.warning]
future_app = release_year_input.set_value(2026).run(timeout=40)
warnings_2026 = [warning.value for warning in future_app.warning]
streamlit_result = {
    "exceptions":len(app_test.exception) + len(future_app.exception), "tabs":tab_labels,
    "year_2020_warning_count":len(warnings_2020),
    "year_2026_warning_count":len(warnings_2026),
    "year_2026_warning":warnings_2026[0] if warnings_2026 else "",
    "status":"PASS" if (
        not app_test.exception and not future_app.exception and len(tab_labels)==4
        and not warnings_2020 and warnings_2026
        and "product support cutoff" in warnings_2026[0]
    ) else "FAIL",
}
assert streamlit_result["status"] == "PASS", streamlit_result
display(pd.DataFrame([streamlit_result]))

2026-08-14 13:04:05.346 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,exceptions,tabs,year_2020_warning_count,year_2026_warning_count,year_2026_warning,status
0,0,"[Overview, Popularity Prediction, Song Cluster...",0,1,The model was trained through 2018. The produc...,PASS


In [5]:
validation = {"pipeline":e2e, "api_direct_prediction_parity":True,
              "prediction_support_policy":{
                  "product_support_end_year":PRODUCT_SUPPORT_END_YEAR,
                  "observed_data_max_year":OBSERVED_DATA_MAX_YEAR,
                  "final_holdout_max_year":FINAL_HOLDOUT_MAX_YEAR,
                  "year_2020":pred_2020.json(), "year_2026":pred_2026.json(),
                  "warning_does_not_change_prediction":True,
                  "status":"PASS"},
              "loaded_estimator":estimator_name,
              "metadata_winner_model":metrics["selection_winner_model"],
              "health":health.json(), "prediction":pred_response.json(),
              "cluster":cluster_response.json(), "recommendation":recommend_response.json(),
              "streamlit":streamlit_result}
validation_path = ROOT / "5.UNG_DUNG" / "validation" / "round4_end_to_end_validation.json"
validation_path.parent.mkdir(parents=True, exist_ok=True)
validation_path.write_text(json.dumps(validation, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", validation_path)
print("ROUND 4 END-TO-END STATUS: PASS")

Saved: D:\Hitradar\hitradar-main\5.UNG_DUNG\validation\round4_end_to_end_validation.json
ROUND 4 END-TO-END STATUS: PASS


## Kết luận

Deployment tải đúng winner đã lock từ Notebook 06, nhận raw inputs, tái tạo features trong pipeline, clip popularity về [0,100], và phục vụ cluster/recommendation từ artifacts Notebook 05. Dự đoán sau 2020 vẫn được phép nhưng được đánh dấu là ngoại suy theo thời gian; cảnh báo không thay đổi giá trị dự đoán. Streamlit có đúng bốn tab: Overview, Popularity Prediction, Song Clustering, Similar Songs.